In [1]:
from ml_recruitment import ML_Recruitment
import os
from numpy import nan

vCaminhoBase = os.path.join("..", 'etl', "output", "dataset_unificado_balanceado.csv")

vML = ML_Recruitment(pCaminhoBase=vCaminhoBase)

# Carregar a base de dados
vML.carregarBase()

# Criar uma coluna de teste com valores NaN
vML.baseDeDados['TesTE de Nome de ColunA'] = nan

# Padronizar a base de dados
vML.padronizarBase(pListaDeColunasParaFillNA=['requisitos_vaga', 'cv_texto', 'TesTE de Nome de ColunA'], pPreencherNulosCom='vazio', pPadronizarColunasTexto=False)

# Treinar o vetorizador textual
vML.treinarVetorizadorTextual(pListaColunas=['requisitos_vaga', 'cv_texto'], pNumeroMaximoFeatures=300)

# Calcular a similaridade textual
vML.baseDeDados['sim_textual'] = vML.calcularSimilaridadeTextual(pListaColunas=['requisitos_vaga', 'cv_texto'])

# Inserir novas colunas
vML.baseDeDados['match_nivel'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_ingles'] = (vML.baseDeDados['nivel_ingles_vaga'] == vML.baseDeDados['nivel_ingles_candidato']).astype(int)
vML.baseDeDados['match_profissional'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_espanhol'] = (vML.baseDeDados['nivel_espanhol_vaga'] == vML.baseDeDados['nivel_espanhol_candidato']).astype(int)
vML.baseDeDados['match_local'] = (vML.baseDeDados['local_vaga'] == vML.baseDeDados['local_candidato']).astype(int)
vML.baseDeDados['match_academico'] = (vML.baseDeDados['nivel_academico_vaga'] == vML.baseDeDados['nivel_academico_candidato']).astype(int)

# Separar features e target
vML.separarFeatureTarget(pColunaTarget='match', pColunasIgnorar=['situacao', 'comentario', 'recrutador', 'vaga_id', 'codigo_candidato', 'nome_candidato', 'conhecimentos_tecnicos', 'teste_de_nome_de_coluna'])

# Criar a pipeline de pré-processamento
vML.criarPipeline(
    pColunasNumericas=['sim_textual', 'match_nivel', 'match_ingles', 'match_profissional', 'match_espanhol', 'match_local', 'match_academico'],
    pColunasCategoricas=['titulo_vaga', 'nivel_profissional_vaga', 'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga', 'nivel_academico_candidato', 'nivel_ingles_candidato', 'nivel_espanhol_candidato', 'nivel_profissional_candidato', 'local_candidato', 'cliente', 'local_vaga'],
    pColunasTexto=['requisitos_vaga', 'cv_texto'],
    pNumeroMaximoFeatures=30
)

# Separar a base de dados em treino e teste
vML.separarTreinoTeste(pProporcaoTreino=0.2, pRandomState=42)

# Executar a pipeline de pré-processamento
vML.executarPipeline(pAplicarEm='Treino_Teste')

# Criar o modelo XGBoost
vTotalPos = vML.target_Treino.sum()
vScalePosWeight = (len(vML.target_Treino) - vTotalPos) / vTotalPos if vTotalPos > 0 else 1

vParametrosXGB = {
    'n_estimators': 100,
    'eval_metric': 'logloss',
    'scale_pos_weight': vScalePosWeight,
    'use_label_encoder': False,
    'early_stopping_rounds': 10,
    'random_state': 42
}

vML.criarModeloXGB(pParametros=vParametrosXGB, pTreinarModelo=False)

# Treinar o modelo XGBoost
vML.treinarModeloXGB()

# Avaliar o modelo XGBoost
print(vML.avaliarModeloXGB(pOutputDict=False))

# Salvar vetorizador, pipeline e modelo
vML.salvarVetorizadorTextual(pCaminhoArquivo=os.path.join("output", "vetorizador_similaridade_textual.pkl"))
vML.salvarPipeline(pCaminhoArquivo=os.path.join("output", "preprocessador_xgb.pkl"))
vML.salvarModeloXGB(pCaminhoArquivo=os.path.join("output", "modelo_xgb.pkl"))


c:\Users\rafae\OneDrive\01_Documentos\13_Projetos\venv_api\Lib\site-packages\xgboost\callback.py:386: UserWarning: [15:16:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


              precision    recall  f1-score   support

           0      0.695     0.831     0.757       639
           1      0.790     0.635     0.704       639

    accuracy                          0.733      1278
   macro avg      0.742     0.733     0.731      1278
weighted avg      0.742     0.733     0.731      1278



In [3]:
vML.prever(pFeatures=vML.features_Teste)

array([1, 0, 0, ..., 1, 0, 1], shape=(1278,))

In [15]:
from ml_recruitment import ML_Recruitment
import pandas as pd

vML = ML_Recruitment()

vBase = pd.DataFrame({
    'requisitos_vaga': ['Python, Machine Learning', 'Java, Spring Boot'],
    'cv_texto': ['Experiência em Python e Machine Learning', 'Conhecimento em Java e Spring Boot'],
    'nivel_profissional_vaga': ['Pleno', 'Sênior'],
    'nivel_ingles_vaga': ['Intermediário', 'Avançado'],
    'nivel_espanhol_vaga': ['Básico', 'Intermediário'],
    'nivel_academico_vaga': ['Superior Completo', 'Mestrado'],
    'local_vaga': ['Remoto', 'Híbrido'],
    'nivel_academico_candidato': ['Superior Completo', 'Mestrado'],
    'nivel_ingles_candidato': ['Intermediário', 'Avançado'],
    'nivel_espanhol_candidato': ['Básico', 'Intermediário'],
    'nivel_profissional_candidato': ['Pleno', 'Sênior'],
    'local_candidato': ['Remoto', 'Híbrido'],
    
})
vML.baseDeDados = vBase

vML.carregarModeloXGB(pCaminhoArquivo=os.path.join("output", "modelo_xgb.pkl"))
vML.carregarPipeline(pCaminhoArquivo=os.path.join("output", "preprocessador_xgb.pkl"))

vML.prever(pFeatures=vBase)

ValueError: columns are missing: {'sim_textual', 'match_espanhol', 'match_ingles', 'titulo_vaga', 'match_nivel', 'match_academico', 'match_local', 'match_profissional', 'cliente'}

In [13]:
print(vML)

ValueError: Features não foram separadas. Use o método 'separarFeatureTarget' primeiro.

In [ ]:
import requests

vJson = {
  "pDados": [
    {
      "nivel_profissional_vaga": "",
      "nivel_profissional_candidato": "",
      "nivel_ingles_vaga": "",
      "nivel_espanhol_vaga": "",
      "nivel_academico_vaga": "",
      "nivel_academico_candidato": "",
      "nivel_ingles_candidato": "",
      "nivel_espanhol_candidato": "",
      "local_vaga": "",
      "local_candidato": "",
      "cliente": "",
      "titulo_vaga": "",
      "requisitos_vaga": "",
      "cv_texto": ""
    }
  ],
  "pCaminhoModelo": "../etl/output/modelo_match_xgb.joblib",
  "pCaminhoPipeline": "../etl/output/preprocessador_xgb.joblib",
  "pCaminhoVetorizador": "../etl/output/vetorizador_sim_textual.joblib"
}

vUrl = "http://localhost:8000/prever"

vResposta = requests.post(vUrl, json=vJson)
print(vResposta.json())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)